In [1]:
!java --version

openjdk 21.0.10 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)


In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

In [4]:
from pyspark import SparkContext
print(SparkContext._active_spark_context)


None


In [5]:
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()

In [6]:
spark = SparkSession.builder.appName("PySpark WorkCount Test").master("local[*]").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 06:54:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
print("Spark Version:", spark.version)

Spark Version: 4.1.1


In [8]:
data = ["hello spark", "hello docker", "hello worker"]
print(data, type(data))

['hello spark', 'hello docker', 'hello worker'] <class 'list'>


In [9]:
rdd = spark.sparkContext.parallelize(data)
print(rdd, type(rdd))

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299 <class 'pyspark.core.rdd.RDD'>


In [12]:
counts = rdd.flatMap(lambda x: x.split()).map(lambda word:(word, 1)).reduceByKey(lambda a,b: a+b)
print(counts, type(counts))

PythonRDD[10] at RDD at PythonRDD.scala:58 <class 'pyspark.core.rdd.PipelinedRDD'>


In [13]:
for word, count in counts.collect():

    print(f"{word}:{count}")

spark:1
worker:1
docker:1
hello:3


In [14]:
rdd = spark.sparkContext.textFile("/opt/spark/works/sample.txt")

## 해당 위치의 파일이 text파일이라서 앞의 함수가 textFile()임 위엔 단순 list라서 parallelize()

In [15]:
counts = rdd.flatMap(lambda x: x.split()).map(lambda word:(word, 1)).reduceByKey(lambda a,b: a+b)

In [16]:
for word, count in counts.collect():

    print(f"{word}:{count}")

name:2
is:2
sooah:3
i:1
am:1
my:2
hi:1


**대소문자 구분함**

In [17]:
df = counts.toDF(["word","count"])

In [18]:
print(df)

DataFrame[word: string, count: bigint]


In [19]:
df.show()

+-----+-----+
| word|count|
+-----+-----+
| name|    2|
|   is|    2|
|sooah|    3|
|    i|    1|
|   am|    1|
|   my|    2|
|   hi|    1|
+-----+-----+



In [ ]:
df.coalesce(1).write.mode("overwrite").csv("/opt/spark/output/wordcount",header=True)

이게 셀 csv로 저장하는 방법

In [22]:
df.coalesce(1).write.mode("overwrite").json("/opt/spark/output/wordcount_json")

이게 json데이터로 저장하는 방법

In [23]:
spark.stop()

다 썼으면 스탑시켜주는게 좋음